# 01 - Thu thập dữ liệu đánh giá Laptop trên FPTShop
Notebook này crawl dữ liệu đánh giá từ FPTShop theo hướng HTTP thuần (requests), chuẩn hóa schema tương thích pipeline EDA và lưu CSV UTF-8.

In [ ]:
%pip install requests pandas

In [1]:
import json
import re
import time
from datetime import datetime
from html import unescape
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode
import hashlib

import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://fptshop.com.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

COMMENT_API_URL = "https://papi.fptshop.com.vn/gw/v1/public/bff-before-order/comment/list"
COMMENT_API_HEADERS = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Order-Channel": "1",
    "Referer": "https://fptshop.com.vn/",
}


def with_query_param(url: str, key: str, value):
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query[key] = str(value)
    return parsed._replace(query=urlencode(query)).geturl()


def collect_product_urls(
    search_url: str,
    max_pages_safety: int = 300,
    max_products: int | None = None,
    hard_cap: int | None = None,
    consecutive_empty_pages: int = 3,
    delay_seconds: float = 0.8,
):
    product_urls = []
    seen = set()
    empty_pages = 0

    for page in range(1, max_pages_safety + 1):
        page_url = with_query_param(search_url, "page", page)
        response = SESSION.get(page_url, timeout=20)
        response.raise_for_status()
        html_text = response.text

        matches = re.findall(r'href="(https://fptshop\.com\.vn/may-tinh-xach-tay/[^"]+|/may-tinh-xach-tay/[^"]+)"', html_text)
        new_count = 0

        for href in matches:
            full_url = urljoin("https://fptshop.com.vn", href.split("?")[0].strip())
            if full_url in seen:
                continue
            seen.add(full_url)
            product_urls.append(full_url)
            new_count += 1

            if max_products is not None and len(product_urls) >= max_products:
                print(f"Đã đạt max_products={max_products}.")
                return product_urls
            if hard_cap is not None and len(product_urls) >= hard_cap:
                print(f"Đã đạt hard_cap={hard_cap} để dừng an toàn.")
                return product_urls

        print(f"Trang {page}: +{new_count} URL sản phẩm, tổng {len(product_urls)}")

        if new_count == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= consecutive_empty_pages:
            print(f"Dừng do {consecutive_empty_pages} trang liên tiếp không có URL mới.")
            break

        time.sleep(delay_seconds)

    return product_urls


def extract_product_code(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')
    match = re.search(r'"upc":\{"code":"([^"]+)"\}', normalized)
    if match:
        return match.group(1)

    match_alt = re.search(r'"sku"\s*:\s*"([0-9]{6,})"', normalized)
    if match_alt:
        return match_alt.group(1)

    return None


def _to_int(value):
    if value is None:
        return None
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None


def _clean_text(value):
    if value is None:
        return None
    cleaned = unescape(str(value))
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or None


def _extract_urls(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()] if value.strip() else []
    urls = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                urls.append(item.strip())
            elif isinstance(item, dict):
                for key in ("url", "image", "src", "thumb", "thumbnail"):
                    img = item.get(key)
                    if isinstance(img, str) and img.strip():
                        urls.append(img.strip())
                        break
    elif isinstance(value, dict):
        for key in ("url", "image", "src", "thumb", "thumbnail"):
            img = value.get(key)
            if isinstance(img, str) and img.strip():
                urls.append(img.strip())
                break
    return urls


def extract_product_metadata(html_text: str, item_code: str | None = None):
    metadata = {
        "item_id": item_code,
        "product_name": None,
        "brand": None,
        "price": None,
        "final_price": None,
        "rating_count_total": None,
        "image_product": None,
    }

    ldjson_blocks = re.findall(
        r'<script[^>]*type="application/ld\\+json"[^>]*>(.*?)</script>',
        html_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    for block in ldjson_blocks:
        raw = unescape(block).strip()
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        candidates = parsed if isinstance(parsed, list) else [parsed]
        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue
            candidate_type = candidate.get("@type")
            type_list = candidate_type if isinstance(candidate_type, list) else [candidate_type]
            if "Product" not in type_list:
                continue

            metadata["product_name"] = metadata["product_name"] or candidate.get("name")

            brand = candidate.get("brand")
            if isinstance(brand, dict):
                metadata["brand"] = metadata["brand"] or brand.get("name")
            elif isinstance(brand, str):
                metadata["brand"] = metadata["brand"] or brand

            offers = candidate.get("offers")
            if isinstance(offers, dict):
                metadata["price"] = metadata["price"] or _to_int(offers.get("price"))
            elif isinstance(offers, list):
                for offer in offers:
                    if isinstance(offer, dict) and offer.get("price") is not None:
                        metadata["price"] = metadata["price"] or _to_int(offer.get("price"))
                        break

            agg = candidate.get("aggregateRating")
            if isinstance(agg, dict):
                metadata["rating_count_total"] = metadata["rating_count_total"] or _to_int(agg.get("reviewCount"))

            image_urls = _extract_urls(candidate.get("image"))
            if image_urls and not metadata["image_product"]:
                metadata["image_product"] = image_urls[0]

    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    if not metadata["product_name"]:
        name_match = re.search(r'"upc":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if name_match:
            metadata["product_name"] = name_match.group(1)

    if not metadata["brand"]:
        brand_match = re.search(r'"brand":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if brand_match:
            metadata["brand"] = brand_match.group(1)

    if metadata["final_price"] is None:
        final_price_match = re.search(r'"productAdvanceInfo":\{.*?"finalPrice":(\d+)', normalized, flags=re.DOTALL)
        if final_price_match:
            metadata["final_price"] = _to_int(final_price_match.group(1))

    if metadata["price"] is None:
        price_match = re.search(r'"productAdvanceInfo":\{.*?"price":(\d+)', normalized, flags=re.DOTALL)
        if price_match:
            metadata["price"] = _to_int(price_match.group(1))

    if metadata["image_product"] is None:
        img_match = re.search(r'"primaryImage":\{"url":"([^"]+)"\}', normalized)
        if img_match:
            metadata["image_product"] = img_match.group(1)

    if metadata["final_price"] is None:
        metadata["final_price"] = metadata["price"]

    return metadata


def extract_comment_payload(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    pattern = r'"comment":(\{.*?"totalCount":\d+.*?\})\s*,\s*"policyProduct"'
    match = re.search(pattern, normalized, flags=re.DOTALL)
    if not match:
        return {}

    comment_raw = match.group(1)
    try:
        return json.loads(comment_raw)
    except json.JSONDecodeError:
        return {}


def fetch_comment_page(content_id: str, skip_count: int = 0, max_result_count: int = 6):
    payload = {
        "content": {"id": str(content_id), "type": "PRODUCT"},
        "state": ["ACTIVE"],
        "skipCount": int(skip_count),
        "maxResultCount": int(max_result_count),
        "sortMethod": 1,
    }
    response = SESSION.post(
        COMMENT_API_URL,
        json=payload,
        headers=COMMENT_API_HEADERS,
        timeout=20,
    )
    response.raise_for_status()

    data = response.json()
    if not isinstance(data, dict):
        raise RuntimeError("Phản hồi API comment không đúng định dạng JSON đối tượng.")
    if data.get("status") != 200:
        message = data.get("error", {}).get("message") or data.get("message") or "Lấy comment thất bại."
        raise RuntimeError(message)

    comment_data = data.get("data") or {}
    items = comment_data.get("items") or []
    total_count = _to_int(comment_data.get("totalCount"))
    return items, total_count


def fetch_product_reviews(product_url: str, page_size: int = 6):
    response = SESSION.get(product_url, timeout=20)
    response.raise_for_status()
    html_text = response.text

    item_code = extract_product_code(html_text)
    product_meta = extract_product_metadata(html_text=html_text, item_code=item_code)

    review_items = []
    total_count = None

    if item_code:
        try:
            review_items, total_count = fetch_comment_page(content_id=item_code, skip_count=0, max_result_count=page_size)
        except Exception as exc:
            print(f"Không lấy được comment API cho {product_url}: {exc}")

    if not review_items:
        comment_payload = extract_comment_payload(html_text)
        review_items = comment_payload.get("items") or []
        total_count = total_count or _to_int(comment_payload.get("totalCount")) or len(review_items)

    if product_meta.get("rating_count_total") is None:
        product_meta["rating_count_total"] = _to_int(total_count or len(review_items))

    return product_meta, review_items, total_count, item_code


def parse_rating_row(rating_row: dict, product_meta: dict, product_url: str, row_index: int):
    created_raw = (
        rating_row.get("createdAt")
        or rating_row.get("createAt")
        or rating_row.get("createdDate")
        or rating_row.get("creationTime")
        or rating_row.get("created_time")
        or rating_row.get("time")
    )
    created_at = pd.to_datetime(created_raw, errors="coerce")
    created_at = created_at.isoformat() if pd.notna(created_at) else None

    user_id = (
        rating_row.get("userId")
        or rating_row.get("customerId")
        or rating_row.get("memberId")
        or rating_row.get("fullName")
        or "anonymous"
    )
    cmt_id = rating_row.get("id") or rating_row.get("commentId") or rating_row.get("_id")
    if cmt_id is None:
        fallback = f"{product_url}_{row_index}_{rating_row.get('content', '')}"
        cmt_id = hashlib.md5(fallback.encode("utf-8")).hexdigest()[:16]

    rating_star = rating_row.get("score") or rating_row.get("rating") or rating_row.get("star")
    comment = (
        rating_row.get("content")
        or rating_row.get("comment")
        or rating_row.get("commentContent")
        or ""
    )
    comment = _clean_text(comment) or ""
    review_title = (
        rating_row.get("title")
        or rating_row.get("subject")
        or rating_row.get("headline")
        or rating_row.get("reviewTitle")
        or None
    )

    verified_raw = (
        rating_row.get("verified_purchase")
        or rating_row.get("verifiedPurchase")
        or rating_row.get("isVerified")
        or rating_row.get("isPurchased")
        or rating_row.get("is_buyer")
    )
    verified_purchase = None
    if verified_raw is not None:
        if isinstance(verified_raw, bool):
            verified_purchase = verified_raw
        elif isinstance(verified_raw, (int, float)):
            verified_purchase = bool(verified_raw)
        elif isinstance(verified_raw, str):
            verified_purchase = verified_raw.strip().lower() in {"true", "1", "yes", "y", "co", "có"}

    product_items = (
        rating_row.get("variantName")
        or rating_row.get("modelName")
        or rating_row.get("attributeValue")
        or ""
    )

    reply_content = None
    reply_created_at = None
    reply_user_id = None
    reply_is_admin = None
    reply_like_count = None
    children = rating_row.get("children") or []
    if isinstance(children, list):
        first_reply = next((child for child in children if isinstance(child, dict)), None)
        if first_reply:
            reply_content = _clean_text(first_reply.get("content"))
            reply_raw = first_reply.get("creationTime") or first_reply.get("createdAt") or first_reply.get("createdDate")
            reply_dt = pd.to_datetime(reply_raw, errors="coerce")
            reply_created_at = reply_dt.isoformat() if pd.notna(reply_dt) else None
            reply_user_id = first_reply.get("fullName") or first_reply.get("userId") or first_reply.get("customerId")
            if first_reply.get("isAdministrator") is not None:
                reply_is_admin = bool(first_reply.get("isAdministrator"))
            reply_like_count = first_reply.get("like") or first_reply.get("helpful") or 0

    image_review_urls = []
    for key in ("images", "imageUrls", "image_urls", "attachments", "pictures", "photos", "media"):
        if key in rating_row:
            image_review_urls = _extract_urls(rating_row.get(key))
            if image_review_urls:
                break
    image_review = " | ".join(image_review_urls) if image_review_urls else None

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": "FPTShop",
        "item_id": product_meta.get("item_id"),
        "product_name": product_meta.get("product_name"),
        "brand": product_meta.get("brand"),
        "price": product_meta.get("price"),
        "final_price": product_meta.get("final_price"),
        "rating_count_total": product_meta.get("rating_count_total"),
        "user_id": str(user_id),
        "rating_star": rating_star,
        "review_title": review_title,
        "comment": comment,
        "verified_purchase": verified_purchase,
        "image_product": product_meta.get("image_product"),
        "image_review": image_review,
        "created_at": created_at,
        "like_count": rating_row.get("like") or rating_row.get("helpful") or 0,
        "product_items": str(product_items).strip(),
        "reply_content": reply_content,
        "reply_created_at": reply_created_at,
        "reply_user_id": reply_user_id,
        "reply_is_admin": reply_is_admin,
        "reply_like_count": reply_like_count,
        "source": "FPTShop"
    }


def crawl_product_reviews(product_url: str, max_reviews: int | None = None, page_size: int = 6, delay_seconds: float = 1.2):
    product_meta, page_reviews, total_count, item_code = fetch_product_reviews(product_url=product_url, page_size=page_size)

    reviews = []
    seen_review_ids = set()
    skip_count = 0
    page_number = 1

    while True:
        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        if not page_reviews:
            print(f"Trang {page_number}: không còn comment nào để thu thập.")
            break

        page_added = 0
        for idx, row in enumerate(page_reviews):
            if max_reviews is not None and len(reviews) >= max_reviews:
                break

            parsed = parse_rating_row(row, product_meta=product_meta, product_url=product_url, row_index=skip_count + idx)
            if not parsed["comment"] and not parsed.get("reply_content"):
                continue
            if parsed["review_id"] in seen_review_ids:
                continue

            seen_review_ids.add(parsed["review_id"])
            reviews.append(parsed)
            page_added += 1

        total_label = total_count if total_count is not None else "?"
        print(f"Trang {page_number}: +{page_added} comment, tổng {len(reviews)}/{total_label}")

        if total_count is not None and len(reviews) >= total_count:
            break

        if len(page_reviews) < page_size:
            break

        skip_count += page_size
        page_number += 1

        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        time.sleep(delay_seconds)
        try:
            page_reviews, page_total = fetch_comment_page(content_id=item_code, skip_count=skip_count, max_result_count=page_size)
            if total_count is None:
                total_count = page_total
        except Exception as exc:
            print(f"Lỗi khi tải trang comment thứ {page_number}: {exc}")
            break

    print(f"{product_url} -> lấy {len(reviews)} comment hiển thị / totalCount={total_count}")
    if total_count is not None and total_count > len(reviews):
        print("Lưu ý: crawler đã phân trang comment, nhưng có thể còn dữ liệu ngoài phạm vi hiển thị nếu API thay đổi.")

    return pd.DataFrame(reviews)

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://fptshop.com.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def with_query_param(url: str, key: str, value):
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query[key] = str(value)
    return parsed._replace(query=urlencode(query)).geturl()

def collect_product_urls(
    search_url: str,
    max_pages_safety: int = 300,
    max_products: int | None = None,
    hard_cap: int | None = None,
    consecutive_empty_pages: int = 3,
    delay_seconds: float = 0.8,
):
    product_urls = []
    seen = set()
    empty_pages = 0

    for page in range(1, max_pages_safety + 1):
        page_url = with_query_param(search_url, "page", page)
        response = SESSION.get(page_url, timeout=20)
        response.raise_for_status()
        html_text = response.text

        matches = re.findall(r'href="(https://fptshop\.com\.vn/may-tinh-xach-tay/[^"]+|/may-tinh-xach-tay/[^"]+)"', html_text)
        new_count = 0

        for href in matches:
            full_url = urljoin("https://fptshop.com.vn", href.split("?")[0].strip())
            if full_url in seen:
                continue
            seen.add(full_url)
            product_urls.append(full_url)
            new_count += 1

            if max_products is not None and len(product_urls) >= max_products:
                print(f"Đã đạt max_products={max_products}.")
                return product_urls
            if hard_cap is not None and len(product_urls) >= hard_cap:
                print(f"Đã đạt hard_cap={hard_cap} để dừng an toàn.")
                return product_urls

        print(f"Trang {page}: +{new_count} URL sản phẩm, tổng {len(product_urls)}")

        if new_count == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= consecutive_empty_pages:
            print(f"Dừng do {consecutive_empty_pages} trang liên tiếp không có URL mới.")
            break

        time.sleep(delay_seconds)

    return product_urls

def extract_product_code(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')
    match = re.search(r'"upc":\{"code":"([^"]+)"\}', normalized)
    if match:
        return match.group(1)

    match_alt = re.search(r'"sku"\s*:\s*"([0-9]{6,})"', normalized)
    if match_alt:
        return match_alt.group(1)

    return None

def _to_int(value):
    if value is None:
        return None
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None

def _extract_urls(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()] if value.strip() else []
    urls = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                urls.append(item.strip())
            elif isinstance(item, dict):
                for key in ("url", "image", "src", "thumb", "thumbnail"):
                    img = item.get(key)
                    if isinstance(img, str) and img.strip():
                        urls.append(img.strip())
                        break
    elif isinstance(value, dict):
        for key in ("url", "image", "src", "thumb", "thumbnail"):
            img = value.get(key)
            if isinstance(img, str) and img.strip():
                urls.append(img.strip())
                break
    return urls

def extract_product_metadata(html_text: str, item_code: str | None = None):
    metadata = {
        "item_id": item_code,
        "product_name": None,
        "brand": None,
        "price": None,
        "final_price": None,
        "rating_count_total": None,
        "image_product": None,
    }

    ldjson_blocks = re.findall(
        r'<script[^>]*type="application/ld\\+json"[^>]*>(.*?)</script>',
        html_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    for block in ldjson_blocks:
        raw = unescape(block).strip()
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        candidates = parsed if isinstance(parsed, list) else [parsed]
        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue
            candidate_type = candidate.get("@type")
            type_list = candidate_type if isinstance(candidate_type, list) else [candidate_type]
            if "Product" not in type_list:
                continue

            metadata["product_name"] = metadata["product_name"] or candidate.get("name")

            brand = candidate.get("brand")
            if isinstance(brand, dict):
                metadata["brand"] = metadata["brand"] or brand.get("name")
            elif isinstance(brand, str):
                metadata["brand"] = metadata["brand"] or brand

            offers = candidate.get("offers")
            if isinstance(offers, dict):
                metadata["price"] = metadata["price"] or _to_int(offers.get("price"))
            elif isinstance(offers, list):
                for offer in offers:
                    if isinstance(offer, dict) and offer.get("price") is not None:
                        metadata["price"] = metadata["price"] or _to_int(offer.get("price"))
                        break

            agg = candidate.get("aggregateRating")
            if isinstance(agg, dict):
                metadata["rating_count_total"] = metadata["rating_count_total"] or _to_int(agg.get("reviewCount"))

            image_urls = _extract_urls(candidate.get("image"))
            if image_urls and not metadata["image_product"]:
                metadata["image_product"] = image_urls[0]

    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    if not metadata["product_name"]:
        name_match = re.search(r'"upc":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if name_match:
            metadata["product_name"] = name_match.group(1)

    if not metadata["brand"]:
        brand_match = re.search(r'"brand":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if brand_match:
            metadata["brand"] = brand_match.group(1)

    if metadata["final_price"] is None:
        final_price_match = re.search(r'"productAdvanceInfo":\{.*?"finalPrice":(\d+)', normalized, flags=re.DOTALL)
        if final_price_match:
            metadata["final_price"] = _to_int(final_price_match.group(1))

    if metadata["price"] is None:
        price_match = re.search(r'"productAdvanceInfo":\{.*?"price":(\d+)', normalized, flags=re.DOTALL)
        if price_match:
            metadata["price"] = _to_int(price_match.group(1))

    if metadata["image_product"] is None:
        img_match = re.search(r'"primaryImage":\{"url":"([^"]+)"\}', normalized)
        if img_match:
            metadata["image_product"] = img_match.group(1)

    if metadata["final_price"] is None:
        metadata["final_price"] = metadata["price"]

    return metadata

def extract_comment_payload(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    pattern = r'"comment":(\{.*?"totalCount":\d+.*?\})\s*,\s*"policyProduct"'
    match = re.search(pattern, normalized, flags=re.DOTALL)
    if not match:
        return {}

    comment_raw = match.group(1)
    try:
        return json.loads(comment_raw)
    except json.JSONDecodeError:
        return {}

def fetch_product_reviews(product_url: str):
    response = SESSION.get(product_url, timeout=20)
    response.raise_for_status()
    html_text = response.text

    item_code = extract_product_code(html_text)
    product_meta = extract_product_metadata(html_text=html_text, item_code=item_code)
    comment_payload = extract_comment_payload(html_text)

    review_items = comment_payload.get("items") or []
    total_count = comment_payload.get("totalCount", len(review_items))

    if product_meta.get("rating_count_total") is None:
        product_meta["rating_count_total"] = _to_int(total_count)

    return product_meta, review_items, total_count

def parse_rating_row(rating_row: dict, product_meta: dict, product_url: str, row_index: int):
    created_raw = (
        rating_row.get("createdAt")
        or rating_row.get("createAt")
        or rating_row.get("createdDate")
        or rating_row.get("created_time")
        or rating_row.get("time")
    )
    created_at = pd.to_datetime(created_raw, errors="coerce")
    created_at = created_at.isoformat() if pd.notna(created_at) else None

    user_id = (
        rating_row.get("userId")
        or rating_row.get("customerId")
        or rating_row.get("memberId")
        or rating_row.get("fullName")
        or "anonymous"
    )
    cmt_id = rating_row.get("id") or rating_row.get("commentId") or rating_row.get("_id")
    if cmt_id is None:
        fallback = f"{product_url}_{row_index}_{rating_row.get('content', '')}"
        cmt_id = hashlib.md5(fallback.encode("utf-8")).hexdigest()[:16]

    rating_star = rating_row.get("score") or rating_row.get("rating") or rating_row.get("star")
    comment = (
        rating_row.get("content")
        or rating_row.get("comment")
        or rating_row.get("commentContent")
        or ""
    )
    review_title = (
        rating_row.get("title")
        or rating_row.get("subject")
        or rating_row.get("headline")
        or rating_row.get("reviewTitle")
        or None
    )

    verified_raw = (
        rating_row.get("verified_purchase")
        or rating_row.get("verifiedPurchase")
        or rating_row.get("isVerified")
        or rating_row.get("isPurchased")
        or rating_row.get("is_buyer")
    )
    verified_purchase = None
    if verified_raw is not None:
        if isinstance(verified_raw, bool):
            verified_purchase = verified_raw
        elif isinstance(verified_raw, (int, float)):
            verified_purchase = bool(verified_raw)
        elif isinstance(verified_raw, str):
            verified_purchase = verified_raw.strip().lower() in {"true", "1", "yes", "y", "co", "có"}

    product_items = (
        rating_row.get("variantName")
        or rating_row.get("modelName")
        or rating_row.get("attributeValue")
        or ""
    )

    image_review_urls = []
    for key in ("images", "imageUrls", "image_urls", "attachments", "pictures", "photos", "media"):
        if key in rating_row:
            image_review_urls = _extract_urls(rating_row.get(key))
            if image_review_urls:
                break
    image_review = " | ".join(image_review_urls) if image_review_urls else None

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": "FPTShop",
        "item_id": product_meta.get("item_id"),
        "product_name": product_meta.get("product_name"),
        "brand": product_meta.get("brand"),
        "price": product_meta.get("price"),
        "final_price": product_meta.get("final_price"),
        "rating_count_total": product_meta.get("rating_count_total"),
        "user_id": str(user_id),
        "rating_star": rating_star,
        "review_title": review_title,
        "comment": str(comment).strip(),
        "verified_purchase": verified_purchase,
        "image_product": product_meta.get("image_product"),
        "image_review": image_review,
        "created_at": created_at,
        "like_count": rating_row.get("like") or rating_row.get("helpful") or 0,
        "product_items": str(product_items).strip(),
        "source": "FPTShop"
    }

def crawl_product_reviews(product_url: str, max_reviews: int | None = None, page_size: int = 50, delay_seconds: float = 1.2):
    product_meta, page_reviews, total_count = fetch_product_reviews(product_url=product_url)

    reviews = []
    seen_review_ids = set()

    for idx, row in enumerate(page_reviews):
        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        parsed = parse_rating_row(row, product_meta=product_meta, product_url=product_url, row_index=idx)
        if not parsed["comment"]:
            continue
        if parsed["review_id"] in seen_review_ids:
            continue

        seen_review_ids.add(parsed["review_id"])
        reviews.append(parsed)

    print(f"{product_url} -> lấy {len(reviews)} review hiển thị / totalCount={total_count}")
    if total_count > len(page_reviews):
        print("Lưu ý: hiện chỉ thu được danh sách review hiển thị từ HTML, chưa phân trang sâu bằng endpoint nội bộ.")

    time.sleep(delay_seconds)
    return pd.DataFrame(reviews)

In [ ]:
search_url = "https://fptshop.com.vn/tim-kiem?s=laptop&sort=noi-bat&categories=may-tinh-xach-tay"
product_urls = collect_product_urls(
    search_url=search_url,
    max_pages_safety=300,
    max_products=None,
    hard_cap=None,
    consecutive_empty_pages=3,
    delay_seconds=0.8
)

display(product_urls[:10])
print(f"Tổng URL sản phẩm laptop đã thu thập: {len(product_urls)}")

if not product_urls:
    raise ValueError("Không lấy được danh sách product_urls từ trang tìm kiếm FPTShop.")

In [ ]:
all_frames = []
total_urls = len(product_urls)

for idx, url in enumerate(product_urls, start=1):
    print(f"[{idx}/{total_urls}] Đang crawl: {url}")
    df_item = crawl_product_reviews(url, max_reviews=None, page_size=6, delay_seconds=1.2)
    if not df_item.empty:
        all_frames.append(df_item)

if not all_frames:
    raise RuntimeError("Không thu được dữ liệu. Kiểm tra lại URL hoặc thử chạy lại với mạng khác.")

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

display(df_raw.head())
display(df_raw.tail())
print(f"Tổng số review thu được: {len(df_raw)}")

In [ ]:
output_path = "data/fptshop_laptop_raw.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu dữ liệu thô vào: {output_path}")

## Ghi chú
Notebook đã chuyển sang gọi API comment riêng của FPTShop để lấy dữ liệu đánh giá theo phân trang (`skipCount`/`maxResultCount`), nên có thể thu thêm comment ở các trang 1, 2, 3, ... thay vì chỉ đọc khối comment đầu tiên trong HTML. Dữ liệu thô vẫn giữ schema cũ và bổ sung metadata để phân tích: product_name, brand, price/final_price, rating_count_total, review_title, verified_purchase, image_product, image_review, cùng reply_content nếu comment có phản hồi từ quản trị viên. Một số cột có thể null vì phụ thuộc dữ liệu FPTShop tại thời điểm crawl.